In [1]:
!pip install pytorch-lightning lightning-bolts wandb torchvision

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.0/330.0 kB 5.1 MB/s eta 0:00:00


In [2]:
import pytorch_lightning as pl
from pytorch_lightning import LightningDataModule
from pl_bolts.models.regression import LogisticRegression
from pl_bolts.callbacks import PrintTableMetricsCallback
from sklearn.model_selection import KFold
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader,Dataset
from pytorch_lightning.loggers import WandbLogger
from torchvision import transforms

from typing import Optional

/opt/conda/lib/python3.7/site-packages/pl_bolts/models/self_supervised/amdim/amdim_module.py:35: UnderReviewWarning: The feature generate_power_seq is currently marked under review. The compatibility with other Lightning projects is not guaranteed and API may change at any time. The API and functionality may change without warning in future releases. More details: https://lightning-bolts.readthedocs.io/en/latest/stability.html
  "lr_options": generate_power_seq(LEARNING_RATE_CIFAR, 11),
/opt/conda/lib/python3.7/site-packages/pl_bolts/models/self_supervised/amdim/amdim_module.py:93: UnderReviewWarning: The feature FeatureMapContrastiveTask is currently marked under review. The compatibility with other Lightning projects is not guaranteed and API may change at any time. The API and functionality may change without warning in future releases. More details: https://lightning-bolts.readthedocs.io/en/latest/stability.html
  contrastive_task: Union[FeatureMapContrastiveTask] = FeatureMapContr

In [3]:
import wandb

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    secret_value_0 = user_secrets.get_secret("wandb_api")
    wandb.login(key=secret_value_0)
except:
    print('If you want to use your W&B account, go to Add-ons -> Secrets and provide your W&B access token. Use the Label name as wandb_api. \nGet your W&B access token from here: https://wandb.ai/authorize')

wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [4]:
RANDOM_SEED=3407
pl.seed_everything(RANDOM_SEED)

3407

## Global Parameters

In [5]:
TEST_PATH="../input/digit-recognizer/test.csv"
TRAIN_PATH="../input/digit-recognizer/train.csv"
NUM_SPLITS=3

INPUT_DIM=784
NUM_CLASSES=10

EPOCHS=100

## Data

In [6]:
# get the normalize transform
def get_transforms():
    train=pd.read_csv(TRAIN_PATH)
    train_features=train.iloc[:,1:].to_numpy()
    train_features=train_features.astype(np.uint8)
    
    # calculate the mean and std for scaled images
    mean=train_features.mean()/255
    std=train_features.std()/255
    
    # transforms
    trans=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((mean,), (std,))
    ])
    return trans
    
trans=get_transforms()

In [7]:
class NumpyDataset(Dataset):
    def __init__(self,x,y,transform=None):
        self.x=x
        self.y=y
        self.transform=transform
        if self.transform:
            self.x=self.transform(self.x)
            self.x=torch.squeeze(self.x,0)
        
    def __len__(self):
        return len(self.x)
    
    def __getitem__(self,idx):
        x,y=self.x[idx],self.y[idx]
        return x,y

In [8]:
class KFoldDataModule(LightningDataModule):
    """
    Modified from https://gist.github.com/ashleve/ac511f08c0d29e74566900fd3efbb3ec
    """
    def __init__(
            self,
            train_path: str = "../input/digit-recognizer/train.csv",
            k: int = 1,  # fold number
            split_seed: int = 12345,  # split needs to be always the same for correct cross validation
            num_splits: int = 10,
            batch_size: int = 256,
            num_workers: int = 0,
            pin_memory: bool = False
        ):
        super().__init__()
        
        # this line allows to access init params with 'self.hparams' attribute
        self.save_hyperparameters(logger=False)

        # num_splits = 10 means our dataset will be split to 10 parts
        # so we train on 90% of the data and validate on 10%
        assert 0 <= self.hparams.k < self.hparams.num_splits, "incorrect fold number"
        
        # data transformations
        self.transforms = None

        self.data_train = None
        self.data_val = None
            
    def get_dataset_full(self,transform):
        train=pd.read_csv(self.hparams.train_path)
        train_features=train.iloc[:,1:].to_numpy()
        train_features=train_features.astype(np.uint8)
        train_labels=train.iloc[:,0].to_numpy()
        return NumpyDataset(train_features,train_labels,transform=transform)

    def setup(self, stage=None, transform=None):
        if not self.data_train and not self.data_val:
            dataset_full = self.get_dataset_full(transform)

            # choose fold to train on
            kf = KFold(n_splits=self.hparams.num_splits, shuffle=True, random_state=self.hparams.split_seed)
            all_splits = [k for k in kf.split(dataset_full)]
            train_indexes, val_indexes = all_splits[self.hparams.k]
            train_indexes, val_indexes = train_indexes.tolist(), val_indexes.tolist()

            (train_x,train_y),(val_x,val_y) = dataset_full[train_indexes], dataset_full[val_indexes]
            self.data_train=NumpyDataset(train_x,train_y)
            self.data_val=NumpyDataset(val_x,val_y)

    def train_dataloader(self):
        return DataLoader(dataset=self.data_train, batch_size=self.hparams.batch_size, num_workers=self.hparams.num_workers,
                          pin_memory=self.hparams.pin_memory, shuffle=True)

    def val_dataloader(self):
        return DataLoader(dataset=self.data_val, batch_size=self.hparams.batch_size, num_workers=self.hparams.num_workers,
                          pin_memory=self.hparams.pin_memory)

In [9]:
dms=[]
for i in range(NUM_SPLITS):
    dms.append(KFoldDataModule(train_path=TRAIN_PATH,k=i,split_seed=RANDOM_SEED,num_splits=NUM_SPLITS))
    dms[-1].prepare_data()
    dms[-1].setup(transform=trans)

In [10]:
def get_test_dataloader(test_path,batch_size=256,num_workers=0,pin_memory=False,transform=None):
    test=pd.read_csv(test_path)
    test_features=test.to_numpy()
    test_features=test_features.astype(np.uint8)
    dummy_labels=np.zeros(len(test_features))
    
    test_dataset=NumpyDataset(test_features,dummy_labels,transform=transform)
    return DataLoader(dataset=test_dataset, batch_size=batch_size, num_workers=num_workers,
                          pin_memory=pin_memory,shuffle=False)

test_dataloader=get_test_dataloader(TEST_PATH,transform=trans)

## Model

In [11]:
class Model(LogisticRegression):
    def __init__(self,*args,**kwargs):
        super().__init__(*args, **kwargs)
        
    def training_step(self,batch,batch_idx):
        train_dict=super().training_step(batch,batch_idx)
        self.log_dict(train_dict["log"],on_epoch=True,on_step=False,prog_bar=True)
        return train_dict
        
    def validation_epoch_end(self, outputs):
        val_dict=super().validation_epoch_end(outputs)
        self.log_dict(val_dict["log"],on_epoch=True,on_step=False,prog_bar=True)
        return val_dict
    
    def predict_step(self,batch,batch_idx):
        x,_=batch # y should be ignored
        # will use soft voting, so return the posibility is fine
        return self(x)

## Train model

In [12]:
def train(fold,dm): 
    callbacks=[]
    checkpoint_callback = ModelCheckpoint(
        dirpath=f"./checkpoints/fold{fold}/",
        filename="epoch_{epoch}-val_loss_{val_ce_loss:.2f}",
        monitor='val_ce_loss', verbose=True,
        save_last=False, save_top_k=1,save_weights_only=False,
        mode='min'
    )        
    callbacks.append(checkpoint_callback)
    
    early_stopping = EarlyStopping('val_acc', patience=10, verbose=True, mode='max')
    callbacks.append(early_stopping)    
    
    wandb_logger = WandbLogger(name=f'logistic_regression_fold{fold}',
                           project='digit_recognizer', 
                           offline=False, 
                           log_model=False)
    
    trainer = pl.Trainer(
                        accelerator='gpu' if torch.cuda.is_available() else "cpu",
                        devices=-1 if torch.cuda.is_available() else None,
                        max_epochs=EPOCHS,
                        callbacks=callbacks,
                        check_val_every_n_epoch=1,
                        logger=wandb_logger,
                        enable_progress_bar=True)
    
    # create new model for each fold
    model = Model(input_dim=INPUT_DIM, num_classes=NUM_CLASSES)
    
    trainer.fit(model, dm)
    
    print(checkpoint_callback.best_model_path, checkpoint_callback.best_model_score)
    wandb.finish()
    return model

In [13]:
models=[]
for irun in range(NUM_SPLITS):
    model=train(irun,dm=dms[irun])
    models.append(model)

wandb: Currently logged in as: xiziyi. Use `wandb login --relogin` to force relogin


/opt/conda/lib/python3.7/site-packages/ipykernel_launcher.py:3: UnderReviewWarning: The feature LogisticRegression is currently marked under review. The compatibility with other Lightning projects is not guaranteed and API may change at any time. The API and functionality may change without warning in future releases. More details: https://lightning-bolts.readthedocs.io/en/latest/stability.html
  This is separate from the ipykernel package so we can avoid doing imports until


Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

/kaggle/working/checkpoints/fold2/epoch_epoch=99-val_loss_val_ce_loss=0.28.ckpt tensor(0.2796, device='cuda:0')


epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train_ce_loss,█▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
trainer/global_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
val_acc,▁▅▆▇▇▇▇▇████████████████████████████████
val_ce_loss,█▄▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,99
train_ce_loss,0.23695
trainer/global_step,10999
val_acc,0.92223
val_ce_loss,0.27962


## Predict

In [14]:
def get_predictions(models,test_dataloader):
    possibility=None
    for irun in range(NUM_SPLITS):
        trainer = pl.Trainer(
                            accelerator='gpu' if torch.cuda.is_available() else "cpu",
                            devices=-1 if torch.cuda.is_available() else None,
                            enable_progress_bar=True)
        if possibility==None:
            possibility = torch.cat(trainer.predict(models[irun], test_dataloader),0)
        else:
            possibility += torch.cat(trainer.predict(models[irun], test_dataloader),0)
    # get argmax
    predictions=torch.argmax(possibility,dim=1)
    return pd.DataFrame.from_dict(
    {
        "ImageId":range(1,len(predictions)+1),
        "label":predictions.cpu().detach().numpy()
    })
    
predictions=get_predictions(models,test_dataloader)
predictions.to_csv("submission.csv",index=False)

Predicting: 0it [00:00, ?it/s]

/opt/conda/lib/python3.7/site-packages/pl_bolts/models/regression/logistic_regression.py:49: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  y_hat = softmax(x)


Predicting: 0it [00:00, ?it/s]

Predicting: 0it [00:00, ?it/s]